# 03 Planning — Plan-as-Tool

LLM 自主决定什么时候列计划。和 save_memory 一样 —— Planning 就是一个工具调用，不是新模式。

**学习点**：工具驱动策略、LLM 自主决策、计划状态管理

In [ ]:
import sys, json, os, shutil
sys.path.insert(0, '..')

from src.agent_framework import Agent

for d in ['agent_memory']:
    if os.path.exists(d):
        shutil.rmtree(d)

print('一切就绪')

## 准备工具和 Agent

三个工具：search_web、make_plan、update_plan

In [ ]:
def search_web(query):
    db = {'特斯拉': '特斯拉股价 $245，上季度 $220', '苹果': '苹果股价 $190，上季度 $175',
         '茅台': '茅台股价 ¥1650', '图灵奖': '图灵奖是计算机领域最高荣誉'}
    return next((v for k, v in db.items() if k in query), '未找到')

_agent_ref = [None]

def _make_plan(task, steps):
    agent = _agent_ref[0]
    if agent and agent.active_plan:
        return '已有活跃计划。请先完成或调用 update_plan。'
    # 去掉 LLM 可能加的数字前缀
    cleaned = []
    for s in steps:
        s = s.strip()
        while s and s[0] in '0123456789.、 ':
            s = s[1:]
        cleaned.append(s.strip())
    plan = {'task': task, 'steps': [{'desc': s, 'status': '○'} for s in cleaned], 'current_step': 0}
    if agent:
        agent.active_plan = plan
    return f'计划：' + ', '.join(f'{i+1}.{s}' for i, s in enumerate(cleaned))

def _update_plan(action, changes=''):
    agent = _agent_ref[0]
    if not agent or not agent.active_plan:
        return '没有活跃计划'
    plan = agent.active_plan
    parts = action.split(maxsplit=1)
    cmd = parts[0]
    try:
        if cmd == 'complete_step':
            idx = int(parts[1]) - 1 if len(parts) > 1 else plan['current_step']
            plan['steps'][idx]['status'] = '✓'
            if idx + 1 < len(plan['steps']):
                plan['current_step'] = idx + 1
                plan['steps'][idx+1]['status'] = '→'
        elif cmd == 'add_step' and changes:
            plan['steps'].append({'desc': changes, 'status': '○'})
    except (ValueError, IndexError):
        return '操作失败'
    return f'计划已更新（当前第 {plan["current_step"]+1} 步）'

DEMO_TOOLS = [
    {'name': 'search_web', 'description': '搜索网页获取信息',
     'parameters': {'type': 'object', 'properties': {'query': {'type': 'string'}}, 'required': ['query']},
     'fn': search_web},
    {'name': 'make_plan', 'description': '当任务复杂需要多步协调时，先制定分步计划再执行。已有计划时请勿重复调用。',
     'parameters': {'type': 'object', 'properties': {'task': {'type': 'string'}, 'steps': {'type': 'array', 'items': {'type': 'string'}}}, 'required': ['task', 'steps']},
     'fn': _make_plan},
    {'name': 'update_plan', 'description': '更新当前计划：complete_step n 标记完成',
     'parameters': {'type': 'object', 'properties': {'action': {'type': 'string'}, 'changes': {'type': 'string'}}, 'required': ['action']},
     'fn': _update_plan},
]

agent = Agent(tools=DEMO_TOOLS, system_prompt='面对需要多步协调的复杂任务时，先调用 make_plan 制定计划。')
_agent_ref[0] = agent
print('Agent 已就绪')

## 复杂任务：LLM 自动调 make_plan

LLM 判断任务复杂度 → 自动创建计划 → 逐步执行

In [ ]:
reply = agent.chat('比较特斯拉和苹果的股价，分析投资价值')
print(f'\n最终回复: {reply[:200]}...' if len(reply) > 200 else f'\n最终回复: {reply}')
print(f'\nActive plan: {json.dumps(agent.active_plan, ensure_ascii=False, indent=2) if agent.active_plan else "无"}')

## 简单任务：不调 make_plan

LLM 判断不需要计划，直接回答。

In [ ]:
agent2 = Agent(tools=DEMO_TOOLS, system_prompt='你是一个AI助手')
_agent_ref[0] = agent2
reply = agent2.chat('你好，今天天气怎么样？')
print(f'回复: {reply[:150]}')
print(f'Active plan: {"无" if not agent2.active_plan else agent2.active_plan["task"]}')

## 防误覆盖

已有计划时 LLM 再调 make_plan → 被拒绝。

In [ ]:
# agent 已有 plan（从上个测试来）
reply = agent.chat('我想重新制定一个关于茅台股价的分析计划')
print(f'回复: {reply[:300]}')
print(f'\nActive plan 是否还在: {"是" if agent.active_plan else "否"}')

## 手动 /plan

CLI 里用 `/plan <任务>` 强制覆盖旧计划。编程模式下等价于直接设置 active_plan。

In [ ]:
agent.active_plan = {
    'task': '分析茅台投资价值',
    'steps': [{'desc': '请先分析任务，制定具体步骤', 'status': '→'}],
    'current_step': 0,
}
reply = agent.chat('请为"分析茅台投资价值"制定具体步骤')
print(f'回复: {reply[:200]}')
print(f'\n计划步骤: {[s["desc"] for s in agent.active_plan["steps"]]}')

shutil.rmtree('agent_memory', ignore_errors=True)
print('演示完成')

## 总结

| | Plan-as-Tool |
|---|---|
| 实现方式 | 一个工具 `make_plan`，和 `save_memory` 完全一样的机制 |
| 谁决定 | LLM 自主判断任务复杂度 |
| Agent 周期 | 不改动，ReAct 循环就是引擎 |
| 防误覆盖 | 上下文提示 + 工具拒绝 |
| 手动兜底 | `/plan` 命令强制覆盖 |

工业上（Claude Code 的 EnterPlanMode）也是这个模式：plan 是一个工具调用，不是另一种执行模式。